In [8]:
# Filters

import numpy as np

def first_order_lpf(fc, fs):
    """Generate first-order low-pass filter coefficients."""
    T = 1/fs
    wc = 2 * np.pi * fc
    
    # Bilinear transform with pre-warping
    alpha = np.tan(wc * T / 2)
    
    # Coefficients (direct form II)
    a1 = (1 - alpha) / (1 + alpha)
    b0 = alpha / (1 + alpha)
    b1 = alpha / (1 + alpha)
    
    # Convert to Q15 fixed-point
    Q15 = 2**15
    return {
        'a1': int(a1 * Q15),
        'b0': int(b0 * Q15),
        'b1': int(b1 * Q15)
    }

def first_order_hpf(fc, fs):
    """Generate first-order high-pass filter coefficients."""
    T = 1/fs
    wc = 2 * np.pi * fc
    
    alpha = np.tan(wc * T / 2)
    
    # High-pass has different numerator
    a1 = (1 - alpha) / (1 + alpha)
    b0 = 1 / (1 + alpha)
    b1 = -1 / (1 + alpha)
    
    Q15 = 2**15
    return {
        'a1': int(a1 * Q15),
        'b0': int(b0 * Q15),
        'b1': int(b1 * Q15)
    }

def peaking_eq(f0, Fs, Q, G_dB):
    """Generate second-order peaking EQ coefficients and convert to Q15."""
    A = 10**(G_dB / 40)
    w0 = 2 * np.pi * f0 / Fs
    alpha = np.sin(w0) / (2 * Q)
    cos_w0 = np.cos(w0)
    
    # Biquad peaking EQ coefficients
    b0 = 1 + alpha * A
    b1 = -2 * cos_w0
    b2 = 1 - alpha * A
    a0 = 1 + alpha / A
    a1 = -2 * cos_w0
    a2 = 1 - alpha / A
    
    # Normalize by a0
    b0 /= a0
    b1 /= a0
    b2 /= a0
    a1 /= a0
    a2 /= a0
    
    Q15 = 2**15
    return {
        'b0': int(b0 * Q15),
        'b1': int(b1 * Q15),
        'b2': int(b2 * Q15),
        'a1': int(a1 * Q15),
        'a2': int(a2 * Q15)
    }

# -------------------------
# Generate filter coefficients
# -------------------------
Fs = 48000

# Low and high shelves
low = first_order_lpf(200, Fs)
high = first_order_hpf(3000, Fs)

print("Low Shelf (200 Hz):")
print(f"  parameter signed [15:0] LOW_A1 = -16'sd{abs(low['a1'])};")
print(f"  parameter signed [15:0] LOW_B0 = 16'sd{low['b0']};")
print(f"  parameter signed [15:0] LOW_B1 = 16'sd{low['b1']};")

print("\nHigh Shelf (3000 Hz):")
print(f"  parameter signed [15:0] HIGH_A1 = -16'sd{abs(high['a1'])};")
print(f"  parameter signed [15:0] HIGH_B0 = 16'sd{high['b0']};")
print(f"  parameter signed [15:0] HIGH_B1 = 16'sd{high['b1']};")

# Presence peaking EQ (~4.5 kHz, adjustable gain)
f0 = 4500  # Hz
Q = 1.0
G_dB = 6.0  # Example boost, can be mapped from fx_presence

presence = peaking_eq(f0, Fs, Q, G_dB)

print("\nPresence (4.5 kHz peaking):")
print(f"  parameter signed [15:0] PRES_B0 = 16'sd{presence['b0']};")
print(f"  parameter signed [15:0] PRES_B1 = 16'sd{presence['b1']};")
print(f"  parameter signed [15:0] PRES_B2 = 16'sd{presence['b2']};")
print(f"  parameter signed [15:0] PRES_A1 = -16'sd{abs(presence['a1'])};")
print(f"  parameter signed [15:0] PRES_A2 = 16'sd{presence['a2']};")


Low Shelf (200 Hz):
  parameter signed [15:0] LOW_A1 = -16'sd31921;
  parameter signed [15:0] LOW_B0 = 16'sd423;
  parameter signed [15:0] LOW_B1 = 16'sd423;

High Shelf (3000 Hz):
  parameter signed [15:0] HIGH_A1 = -16'sd21894;
  parameter signed [15:0] HIGH_B0 = 16'sd27331;
  parameter signed [15:0] HIGH_B1 = 16'sd-27331;

Presence (4.5 kHz peaking):
  parameter signed [15:0] PRES_B0 = 16'sd38127;
  parameter signed [15:0] PRES_B1 = 16'sd-45536;
  parameter signed [15:0] PRES_B2 = 16'sd16638;
  parameter signed [15:0] PRES_A1 = -16'sd45536;
  parameter signed [15:0] PRES_A2 = 16'sd21997;
The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


In [10]:
import numpy as np

Q15 = 2**15

def q15(x):
    return int(np.round(x * Q15))

# -------------------------------------------------
# 1st-order shelf BASE filters
# y[n] = A*x[n] + B*y[n-1]
# -------------------------------------------------
def first_order_lpf_base(fc, fs):
    w = np.tan(np.pi * fc / fs)
    A = w / (1 + w)
    B = (1 - w) / (1 + w)
    return q15(A), q15(B)

def first_order_hpf_base(fc, fs):
    w = np.tan(np.pi * fc / fs)
    A = 1 / (1 + w)
    B = (1 - w) / (1 + w)
    return q15(A), q15(B)

# -------------------------------------------------
# RBJ peaking EQ (biquad)
# y[n] = Σb*x − Σa*y
# -------------------------------------------------
def peaking_eq(f0, fs, Q, gain_db):
    A  = 10**(gain_db / 40)
    w0 = 2 * np.pi * f0 / fs
    alpha = np.sin(w0) / (2 * Q)
    c = np.cos(w0)

    b0 = 1 + alpha * A
    b1 = -2 * c
    b2 = 1 - alpha * A
    a0 = 1 + alpha / A
    a1 = -2 * c
    a2 = 1 - alpha / A

    # normalize
    b0 /= a0
    b1 /= a0
    b2 /= a0
    a1 /= a0
    a2 /= a0

    return (
        q15(b0),
        q15(b1),
        q15(b2),
        q15(a1),
        q15(a2),
    )

# -------------------------------------------------
# Generate coefficients
# -------------------------------------------------
Fs = 48_000

LOW_FC  = 200
HIGH_FC = 3000
MID_F0  = 1200
MID_Q   = 1.0
MID_DB  = 6.0   # reference boost; gain knob scales band

low_A, low_B     = first_order_lpf_base(LOW_FC, Fs)
high_A, high_B   = first_order_hpf_base(HIGH_FC, Fs)
mid_b0, mid_b1, mid_b2, mid_a1, mid_a2 = peaking_eq(
    MID_F0, Fs, MID_Q, MID_DB
)

# -------------------------------------------------
# Print Verilog params (drop-in)
# -------------------------------------------------
print("// ------------------------------------------------------------")
print("// AUTO-GENERATED EQ COEFFICIENTS (Q15)")
print("// ------------------------------------------------------------\n")

print("// Low shelf (~200 Hz)")
print(f"localparam signed [15:0] LOW_A  = 16'sd{low_A};")
print(f"localparam signed [15:0] LOW_B  = 16'sd{low_B};\n")

print("// High shelf (~3 kHz)")
print(f"localparam signed [15:0] HIGH_A = 16'sd{high_A};")
print(f"localparam signed [15:0] HIGH_B = 16'sd{high_B};\n")

print("// Mid peaking (~1.2 kHz)")
print(f"localparam signed [15:0] MID_B0 = 16'sd{mid_b0};")
print(f"localparam signed [15:0] MID_B1 = 16'sd{mid_b1};")
print(f"localparam signed [15:0] MID_B2 = 16'sd{mid_b2};")
print(f"localparam signed [15:0] MID_A1 = -16'sd{abs(mid_a1)};")
print(f"localparam signed [15:0] MID_A2 = 16'sd{mid_a2};")


// ------------------------------------------------------------
// AUTO-GENERATED EQ COEFFICIENTS (Q15)
// ------------------------------------------------------------

// Low shelf (~200 Hz)
localparam signed [15:0] LOW_A  = 16'sd423;
localparam signed [15:0] LOW_B  = 16'sd31921;

// High shelf (~3 kHz)
localparam signed [15:0] HIGH_A = 16'sd27331;
localparam signed [15:0] HIGH_B = 16'sd21895;

// Mid peaking (~1.2 kHz)
localparam signed [15:0] MID_B0 = 16'sd34479;
localparam signed [15:0] MID_B1 = 16'sd-61333;
localparam signed [15:0] MID_B2 = 16'sd27618;
localparam signed [15:0] MID_A1 = -16'sd61333;
localparam signed [15:0] MID_A2 = 16'sd29329;


# Math Formula and design
https://arachnoid.com/BiQuadDesigner/index.html

In [ ]:
# Convert from Decimla to Fixed Point

# Format Q15

Q15 = 2**15

def q15(x):
    return int(np.round(x * Q15))

# Coeficients for 200Hz Low Shelf
a1 = -1.95683800
a2 = 0.95775001
b0 = 1.00127956
b1 = -1.95678236
b2 = 0.95652609

print("// Low shelf (~200 Hz)")

# Coeficients for 2000Hz High Shelf
a1 = -1.54868561
a2 = 0.63323578
b0 = 1.10792936
b1 = -1.74281218
b2 = 0.71943299
